In [3]:
import json
import time
from pathlib import Path
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
import pandas as pd

PROCESSED_PATH = Path("../docs/processed")
CHROMA_PATH = Path("../chroma_db")
CHROMA_PATH.mkdir(exist_ok=True)

print("ChromaDB version:", chromadb.__version__)

/Users/heshangamage/ws02-devassist/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChromaDB version: 1.5.9


In [4]:
with open(PROCESSED_PATH / "chunks.json", 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")
print(f"\nSample chunk:")
print(f"  Page: {chunks[0]['metadata']['page_title']}")
print(f"  Section: {chunks[0]['metadata']['section']}")
print(f"  Text preview: {chunks[0]['text'][:200]}")

Loaded 3127 chunks

Sample chunk:
  Page: Key Concepts
  Section: General
  Text preview: # Key Concepts

| **Concept** | **Description** | | ---------------------------------- | ------------------------------------------------------------ | | API | An API (Application Programming Interfac


In [5]:
# all-MiniLM-L6-v2 — you already know this model from your clustering project
# Fast, lightweight, good at semantic similarity
# Will download on first run (~80MB), cached after that

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Quick test
test_sentences = [
    "How do I authenticate with OAuth2?",
    "WSO2 API Manager security configuration",
    "Python code example for API creation"
]

embeddings = model.encode(test_sentences)
print(f"Model loaded. Embedding dimension: {embeddings.shape[1]}")
print(f"Test embeddings shape: {embeddings.shape}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8257.92it/s]


Model loaded. Embedding dimension: 384
Test embeddings shape: (3, 384)


In [6]:
# Create a persistent ChromaDB client
# Persistent means it saves to disk — survives between sessions
client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# Delete existing collection if rebuilding
try:
    client.delete_collection("wso2_apim_docs")
    print("Deleted existing collection")
except:
    print("No existing collection to delete")

# Create the sentence transformer embedding function
# ChromaDB will use this automatically when adding and querying
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create collection
collection = client.create_collection(
    name="wso2_apim_docs",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # cosine similarity for text
)

print(f"Collection created: wso2_apim_docs")
print(f"Distance metric: cosine")

No existing collection to delete


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6867.60it/s]


Collection created: wso2_apim_docs
Distance metric: cosine


In [7]:
BATCH_SIZE = 100

texts = [c['text'] for c in chunks]
metadatas = [c['metadata'] for c in chunks]
ids = [f"chunk_{i}" for i in range(len(chunks))]

print(f"Adding {len(chunks)} chunks in batches of {BATCH_SIZE}...")
print("This will take 3-5 minutes.\n")

start_time = time.time()

for i in range(0, len(chunks), BATCH_SIZE):
    batch_end = min(i + BATCH_SIZE, len(chunks))
    
    batch_texts = texts[i:batch_end]
    batch_meta = metadatas[i:batch_end]
    batch_ids = ids[i:batch_end]
    
    # Clean metadata — ChromaDB only accepts str, int, float, bool
    clean_meta = []
    for m in batch_meta:
        clean_meta.append({
            'source_file': str(m.get('source_file', '')),
            'page_title': str(m.get('page_title', '')),
            'section': str(m.get('section', '')),
            'folder': str(m.get('folder', '')),
            'chunk_index': int(m.get('chunk_index', 0)),
            'char_count': int(m.get('char_count', 0))
        })
    
    collection.add(
        documents=batch_texts,
        metadatas=clean_meta,
        ids=batch_ids
    )
    
    progress = batch_end / len(chunks) * 100
    elapsed = time.time() - start_time
    print(f"  {batch_end}/{len(chunks)} chunks ({progress:.0f}%) — {elapsed:.0f}s elapsed")

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.0f}s")
print(f"Collection size: {collection.count()} chunks")

Adding 3127 chunks in batches of 100...
This will take 3-5 minutes.

  100/3127 chunks (3%) — 1s elapsed
  200/3127 chunks (6%) — 2s elapsed
  300/3127 chunks (10%) — 2s elapsed
  400/3127 chunks (13%) — 3s elapsed
  500/3127 chunks (16%) — 4s elapsed
  600/3127 chunks (19%) — 5s elapsed
  700/3127 chunks (22%) — 5s elapsed
  800/3127 chunks (26%) — 6s elapsed
  900/3127 chunks (29%) — 7s elapsed
  1000/3127 chunks (32%) — 8s elapsed
  1100/3127 chunks (35%) — 8s elapsed
  1200/3127 chunks (38%) — 9s elapsed
  1300/3127 chunks (42%) — 10s elapsed
  1400/3127 chunks (45%) — 11s elapsed
  1500/3127 chunks (48%) — 11s elapsed
  1600/3127 chunks (51%) — 12s elapsed
  1700/3127 chunks (54%) — 13s elapsed
  1800/3127 chunks (58%) — 13s elapsed
  1900/3127 chunks (61%) — 14s elapsed
  2000/3127 chunks (64%) — 15s elapsed
  2100/3127 chunks (67%) — 16s elapsed
  2200/3127 chunks (70%) — 16s elapsed
  2300/3127 chunks (74%) — 17s elapsed
  2400/3127 chunks (77%) — 18s elapsed
  2500/3127 chunks

In [8]:
def search_docs(query, n_results=5):
    """Search the vector database for relevant chunks."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    
    hits = []
    for i in range(len(results['documents'][0])):
        hits.append({
            'text': results['documents'][0][i],
            'page': results['metadatas'][0][i]['page_title'],
            'section': results['metadatas'][0][i]['section'],
            'folder': results['metadatas'][0][i]['folder'],
            'distance': results['distances'][0][i]
        })
    
    return hits


# Test with 4 different questions
test_queries = [
    "How do I set up OAuth2 authentication?",
    "What is the MCP gateway in WSO2?",
    "How to create an API in WSO2 API Manager?",
    "Rate limiting and throttling policies"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    hits = search_docs(query, n_results=3)
    for i, hit in enumerate(hits):
        print(f"\n  Result {i+1} (distance: {hit['distance']:.3f})")
        print(f"  Page: {hit['page']}")
        print(f"  Section: {hit['section']}")
        print(f"  Preview: {hit['text'][:200]}...")


Query: How do I set up OAuth2 authentication?

  Result 1 (distance: 0.390)
  Page: secure-apis-using-basic-authentication
  Section: Enabling Basic Authentication for an API
  Preview: # secure-apis-using-basic-authentication

## Enabling Basic Authentication for an API
Basic Authentication is an API level configuration. Please sign in to the API Publisher and click on the API that ...

  Result 2 (distance: 0.414)
  Page: Secure Endpoint with OAuth 2.0
  Section: General
  Preview: # Secure Endpoint with OAuth 2.0

[](/assets/img/design/endpoints/endpoint-security/endpoints-save-button.png)

    !!! info
        The Endpoint Auth Type selected should match with the authenticatio...

  Result 3 (distance: 0.423)
  Page: Kerberos OAuth2 Grant
  Section: General
  Preview: # Kerberos OAuth2 Grant

- **OAuth Client Key** - This is the client key of the service provider, which will be checked for authentication by the Identity Server before providing the access token. - *...

Query: What

In [9]:
# More systematic quality check
print("Retrieval quality check\n")

quality_tests = [
    {
        'query': 'OAuth2 token authentication',
        'expected_keywords': ['oauth', 'token', 'authentication', 'security']
    },
    {
        'query': 'MCP gateway model context protocol',
        'expected_keywords': ['mcp', 'model context', 'gateway', 'ai']
    },
    {
        'query': 'create REST API endpoint',
        'expected_keywords': ['api', 'rest', 'create', 'design']
    },
    {
        'query': 'rate limiting throttling',
        'expected_keywords': ['rate', 'throttl', 'limit', 'policy']
    }
]

scores = []
for test in quality_tests:
    hits = search_docs(test['query'], n_results=3)
    
    # Check if expected keywords appear in top 3 results
    combined_text = ' '.join([h['text'].lower() for h in hits])
    keyword_hits = sum(1 for kw in test['expected_keywords'] 
                      if kw.lower() in combined_text)
    score = keyword_hits / len(test['expected_keywords'])
    scores.append(score)
    
    print(f"Query: {test['query']}")
    print(f"  Keywords found: {keyword_hits}/{len(test['expected_keywords'])} ({score*100:.0f}%)")
    avg_distance = sum(h['distance'] for h in hits) / len(hits)
    print(f"  Avg distance: {avg_distance:.3f}")
    print()

print(f"Overall retrieval score: {sum(scores)/len(scores)*100:.0f}%")

Retrieval quality check

Query: OAuth2 token authentication
  Keywords found: 4/4 (100%)
  Avg distance: 0.415

Query: MCP gateway model context protocol
  Keywords found: 4/4 (100%)
  Avg distance: 0.404

Query: create REST API endpoint
  Keywords found: 4/4 (100%)
  Avg distance: 0.365

Query: rate limiting throttling
  Keywords found: 4/4 (100%)
  Avg distance: 0.341

Overall retrieval score: 100%


In [10]:
# Save metadata about the collection for the app to use
collection_info = {
    'total_chunks': collection.count(),
    'model': 'all-MiniLM-L6-v2',
    'embedding_dim': 384,
    'distance_metric': 'cosine',
    'folders_indexed': [
        'get-started', 'api-design-manage', 'api-gateway',
        'api-security', 'ai-gateway', 'install-and-setup',
        'tutorials', 'use-cases'
    ],
    'chroma_path': str(CHROMA_PATH)
}

with open(PROCESSED_PATH / "collection_info.json", 'w') as f:
    json.dump(collection_info, f, indent=2)

print("Collection info saved.")
print(json.dumps(collection_info, indent=2))

Collection info saved.
{
  "total_chunks": 3127,
  "model": "all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "distance_metric": "cosine",
  "folders_indexed": [
    "get-started",
    "api-design-manage",
    "api-gateway",
    "api-security",
    "ai-gateway",
    "install-and-setup",
    "tutorials",
    "use-cases"
  ],
  "chroma_path": "../chroma_db"
}
